<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/NR_Eng_Rule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
import pandas as pd

# Raw GitHub file URL
#url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/LTE_01_03_2026%20-%20Lite.xlsx"
url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Raw%20Data.xlsx"

# Load Excel file
df = pd.read_excel(url)

# Show first few rows
df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,sec_power
0,101,1011,1,1,2026-03-01,2026-03-01 00:00,61.20,583.70
1,101,1011,1,2,2026-03-01,2026-03-01 00:15,67.31,586.91


In [57]:
url1 = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Site%20information.xlsx"

# Load into dataframe
info_5g = pd.read_excel(url1)
info_5g.head(2)

,#,Site_ID,Site Name,aau_count
0,1,101,AIRPORT_MAHE,3
1,3,103,ANSE_AUX_PIN,3


In [58]:
df["aau_count"] = 0

df["bbu_bp"] = 0
df["max_5g_traffic"] = 0
df["bbu_power_band"] = 0
df["bbu_extra_power"] = 0

df["g1a_bp"] = 0
df["g1a_power_band"] = 0
df["g1a_extra_power"] = 0

df["aau_bp"] = 500
df["aau_power_band"] = 0
df["aau_extra_power"] = 0
df["calc_5g_sec_power"] = 0
df["site_5g_power"] = 0

df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,sec_power,aau_count,bbu_bp,max_5g_traffic,bbu_power_band,bbu_extra_power,g1a_bp,g1a_power_band,g1a_extra_power,aau_bp,aau_power_band,aau_extra_power,calc_5g_sec_power
0,101,1011,1,1,2026-03-01,2026-03-01 00:00,61.20,583.70,0,0,0,0,0,0,0,0,500,0,0,0
1,101,1011,1,2,2026-03-01,2026-03-01 00:15,67.31,586.91,0,0,0,0,0,0,0,0,500,0,0,0


In [59]:
aau_map = dict(zip(info_5g["Site_ID"], info_5g["aau_count"]))
df["aau_count"] = df["Site_ID"].map(aau_map).fillna(0).astype(int)
df[["Site_ID", "aau_count"]].head(2)

,Site_ID,aau_count
0,101,3
1,101,3


In [60]:
import numpy as np
df["bbu_bp"] = np.ceil((75 / df["aau_count"]) * 100) / 100
df[["aau_count", "bbu_bp"]].head(2)

,aau_count,bbu_bp
0,3,25.0
1,3,25.0


In [61]:
df["max_5g_traffic"] = np.select(

    [df["traffic_load_mbps"] < 60,
     df["traffic_load_mbps"] < 200,
     df["traffic_load_mbps"] < 400,
     df["traffic_load_mbps"] < 800,
     df["traffic_load_mbps"] < 1200,
     df["traffic_load_mbps"] < 1300],

    [60, 200, 400, 800, 1200, 1300],

    default=1300
)

df[["traffic_load_mbps", "max_5g_traffic"]].head(2)

,traffic_load_mbps,max_5g_traffic
0,61.20,200
1,67.31,200


In [62]:
df["bbu_power_band"] = np.select(

    [df["traffic_load_mbps"] < 60, df["traffic_load_mbps"] < 200,
     df["traffic_load_mbps"] < 400, df["traffic_load_mbps"] < 800,
     df["traffic_load_mbps"] < 1200, df["traffic_load_mbps"] < 1300],
    [2, 4, 6, 8, 10, 12],
    default=14
)
df[["traffic_load_mbps", "bbu_power_band"]].head(2)

,traffic_load_mbps,bbu_power_band
0,61.20,4
1,67.31,4


In [63]:
df["bbu_extra_power"] = np.ceil(

    (((df["traffic_load_mbps"] / df["max_5g_traffic"])
      * df["bbu_power_band"]) / df["aau_count"]) * 100

) / 100

df[["traffic_load_mbps", "max_5g_traffic",
    "bbu_power_band", "aau_count",
    "bbu_extra_power"]].head(2)

,traffic_load_mbps,max_5g_traffic,bbu_power_band,aau_count,bbu_extra_power
0,61.20,200,4,3,0.41
1,67.31,200,4,3,0.45


In [64]:
#output_df = df[["Site_ID","Cell_ID", "trigger_ID","datetime","bbu_extra_power"]].to_excel("Filtered_Output.xlsx", index=False)

In [65]:
df["g1a_bp"] = np.ceil((80 / df["aau_count"]) * 100) / 100
df[["aau_count", "g1a_bp"]].head(2)

,aau_count,g1a_bp
0,3,26.67
1,3,26.67


In [66]:
df["g1a_power_band"] = np.select(

    [df["traffic_load_mbps"] < 60, df["traffic_load_mbps"] < 200,
     df["traffic_load_mbps"] < 400, df["traffic_load_mbps"] < 800,
     df["traffic_load_mbps"] < 1200, df["traffic_load_mbps"] < 1300],

    [5, 10, 15, 20, 25, 30],

    default=35
)

df[["traffic_load_mbps", "g1a_power_band"]].head(2)

,traffic_load_mbps,g1a_power_band
0,61.20,10
1,67.31,10


In [67]:
df["g1a_extra_power"] = np.ceil(

    (((df["traffic_load_mbps"] / df["max_5g_traffic"])
      * df["g1a_power_band"]) / df["aau_count"]) * 100

) / 100

df[["traffic_load_mbps", "max_5g_traffic",
    "g1a_power_band", "aau_count",
    "g1a_extra_power"]].head(2)

,traffic_load_mbps,max_5g_traffic,g1a_power_band,aau_count,g1a_extra_power
0,61.20,200,10,3,1.02
1,67.31,200,10,3,1.13


In [68]:
df["aau_power_band"] = np.select(

    [df["traffic_load_mbps"] < 60, df["traffic_load_mbps"] < 200,
     df["traffic_load_mbps"] < 400, df["traffic_load_mbps"] < 800,
     df["traffic_load_mbps"] < 1200, df["traffic_load_mbps"] < 1300],
    [50, 100, 150, 200, 250, 300],
    default=350
)
df[["traffic_load_mbps", "aau_power_band"]].head(2)

,traffic_load_mbps,aau_power_band
0,61.20,100
1,67.31,100


In [69]:
print(df.columns)

Index(['Site_ID', 'Cell_ID', 'Sector_ID', 'trigger_ID', 'date', 'datetime',
       'traffic_load_mbps', 'sec_power', 'aau_count', 'bbu_bp',
       'max_5g_traffic', 'bbu_power_band', 'bbu_extra_power', 'g1a_bp',
       'g1a_power_band', 'g1a_extra_power', 'aau_bp', 'aau_power_band',
       'aau_extra_power', 'calc_5g_sec_power'],
      dtype='object')


In [70]:
df["aau_extra_power"] = np.ceil(

    ((df["traffic_load_mbps"] / df["max_5g_traffic"])
     * df["aau_power_band"]) * 100

) / 100

df[["traffic_load_mbps", "max_5g_traffic",
    "aau_power_band", "aau_extra_power"]].head(2)

,traffic_load_mbps,max_5g_traffic,aau_power_band,aau_extra_power
0,61.20,200,100,30.60
1,67.31,200,100,33.66


In [73]:
df["calc_5g_sec_power"] = (

    df["bbu_bp"] + df["bbu_extra_power"] +
    df["g1a_bp"] + df["g1a_extra_power"] +
    df["aau_bp"] + df["aau_extra_power"]

)

df[["calc_5g_sec_power"]].head(2)

,calc_5g_sec_power
0,583.70
1,586.91


In [72]:
output_df = df[["Site_ID","Cell_ID", "trigger_ID","datetime","calc_5g_sec_power"]].to_excel("Filtered_Output.xlsx", index=False)

In [ ]:
print("Done")